<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
# Setup — repo root + data
import os
import numpy as np
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        !git clone https://github.com/TheAlishbahWaheed/flyrank-ml-internship.git
    os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# is_declining_label is used ONLY below to sanity-check signals and to review the
# top of the queue by eye. It is never an input to the score, reason code, or action.
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Shape:", df.shape)
print("Base decline rate (whole slice):", round(df["is_declining_label"].mean(), 3))

Shape: (30000, 45)
Base decline rate (whole slice): 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before writing a rule, I check the two signals it leans on. My lane is **Refresh / Content
Opportunity Scoring** — the question is *which pages should a human look at first?* FlyRank's
live product already answers a version of this with hand-written flags (`needs_ctr_fix`,
`is_quick_win`, refresh flags). Those flags aren't in my data, but the signals behind them are,
so I test whether those signals actually behave the way the flags assume — using
`is_declining_label` (built from `trend_direction`, never a score input) as the honest check.

### Signal check 1 — staleness, behind the refresh flags

**Claim:** "Pages that haven't been updated in a long time are more likely to be declining."
This is the logic behind FlyRank's refresh flags. I bucket `days_since_last_update` into the
same bands as `freshness_tier` and compare the decline rate per band.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Signal 1: staleness vs decline rate, bucketed, with n
sig1 = (
    df.groupby("freshness_tier")
    .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
    .reindex(["0-30", "31-90", "91-180", "181+"])
)
sig1["decline_rate"] = sig1["decline_rate"].round(3)
print(sig1)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471


In [9]:
# Signal 2: visibility/volume vs decline rate, bucketed, with n
sig2 = (
    df.groupby("impression_tier")
    .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
    .reindex(["low", "moderate", "good", "excellent"])
)
sig2["decline_rate"] = sig2["decline_rate"].round(3)
print(sig2)

                     n  decline_rate
impression_tier                     
low              11248         0.454
moderate         10469         0.615
good              7205         0.586
excellent         1078         0.462


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Build the score, reason code, action label; rank; write the CSV
stale = (df["freshness_tier"] == "91-180").astype(int)
visible = df["impression_tier"].isin(["moderate", "good"]).astype(int)

df["score"] = stale * visible * df["impressions_90d"]
flagged = (stale * visible) == 1
df["reason_code"] = flagged.map({True: "stale_and_visible", False: "not_flagged"})
df["action"] = flagged.map({True: "refresh_priority", False: "monitor"})

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked) + 1)

output_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "impressions_90d", "days_since_last_update", "freshness_tier", "impression_tier",
    "avg_position", "ctr", "content_type", "is_declining_label",
]

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
ranked[output_cols].to_csv(out_path, index=False)

print("Flagged rows (refresh_priority):", int(flagged.sum()), "/", len(df))
print("Wrote:", out_path, "-", len(ranked), "rows")

# Honest sanity metric: precision@K vs base rate (not required by the card, but cheap and honest)
base_rate = df["is_declining_label"].mean()
for k in [10, 20, 50, 100]:
    p = ranked.head(k)["is_declining_label"].mean()
    print(f"precision@{k}: {p:.3f}   (base rate: {base_rate:.3f})")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Flagged rows (refresh_priority): 6725 / 30000
Wrote: work/outputs/baseline_action_score.csv - 30000 rows
precision@10: 0.500   (base rate: 0.542)
precision@20: 0.600   (base rate: 0.542)
precision@50: 0.700   (base rate: 0.542)
precision@100: 0.650   (base rate: 0.542)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
top10 = ranked.head(10)[output_cols]
top10
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,rank,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,freshness_tier,impression_tier,avg_position,ctr,content_type,is_declining_label
0,1,content_8f39d99dfcfc,client_6208ef0f77,29981,stale_and_visible,refresh_priority,29981,104,91-180,good,34.3,0.05,keyword article,1
1,2,content_b3fccc13da09,client_6208ef0f77,29890,stale_and_visible,refresh_priority,29890,104,91-180,good,40.2,0.03,keyword article,0
2,3,content_88e1880bd3ab,client_19581e27de,29850,stale_and_visible,refresh_priority,29850,104,91-180,good,5.6,0.07,keyword article,0
3,4,content_9331e5f7eeab,client_19581e27de,29828,stale_and_visible,refresh_priority,29828,104,91-180,good,6.7,0.07,keyword article,0
4,5,content_83163890c43a,client_19581e27de,29822,stale_and_visible,refresh_priority,29822,104,91-180,good,3.1,0.44,keyword article,1
5,6,content_e859812ce999,client_19581e27de,29760,stale_and_visible,refresh_priority,29760,104,91-180,good,6.1,0.05,keyword article,1
6,7,content_afd71ac0a398,client_6208ef0f77,29751,stale_and_visible,refresh_priority,29751,104,91-180,good,38.6,0.09,keyword article,0
7,8,content_110a32997057,client_19581e27de,29717,stale_and_visible,refresh_priority,29717,104,91-180,good,1.5,0.74,keyword article,1
8,9,content_593e2753a70c,client_19581e27de,29708,stale_and_visible,refresh_priority,29708,104,91-180,good,7.6,0.12,keyword article,0
9,10,content_d636f1cf9880,client_6208ef0f77,29697,stale_and_visible,refresh_priority,29697,104,91-180,good,36.3,0.10,keyword article,1


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Weak picks: flagged, ranks well already (page 1), and the audit label says not declining
top50 = ranked.head(50)
weak = top50[(top50["avg_position"] > 0) & (top50["avg_position"] < 10) & (top50["is_declining_label"] == 0)]
print("Weak picks in top 50:", len(weak), "/ 50")
weak[["rank", "content_id", "avg_position", "ctr", "is_declining_label"]]
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weak picks in top 50: 8 / 50


,rank,content_id,avg_position,ctr,is_declining_label
2,3,content_88e1880bd3ab,5.6,0.07,0
3,4,content_9331e5f7eeab,6.7,0.07,0
8,9,content_593e2753a70c,7.6,0.12,0
10,11,content_00a44e45d37c,8.2,0.03,0
24,25,content_c86dd0f724c7,6.6,0.25,0
31,32,content_f4d5b8881d43,5.4,0.12,0
32,33,content_c50ae59fc471,7.6,1.22,0
40,41,content_51358f7a942d,3.5,0.23,0


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.